# Experiment: Shock Meta XGBoost Optuna Lab

Objective: run a controlled Optuna workflow for the BTCUSD 1h shock-meta XGBoost pipeline.

This notebook uses `config/optuna/optuna_shock_meta_xgboost.yaml` as the Optuna spec and `config/experiments/btcusd_1h_shock_meta_xgboost.yaml` as the fixed base pipeline. The search space is intentionally small and selector-based so feature window changes do not break model inputs.


## Guardrails

- Keep `RUN_OPTUNA = False` until the feature preview and baseline config checks pass.
- Treat Optuna results as in-sample research candidates, not production evidence.
- Prefer stability across folds over a single high Sharpe trial.
- Do not expand the search space before reviewing trial count, trade count, and drawdown constraints.


In [ ]:
# Setup: imports, paths, and reproducibility
from __future__ import annotations

from copy import deepcopy
from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd
import yaml
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

repo_root = Path.cwd().resolve()
if not (repo_root / 'src').exists():
    repo_root = next(parent for parent in Path.cwd().resolve().parents if (parent / 'src').exists())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.experiments import runner as runner_mod
from src.experiments.optuna_search import (
    normalize_objective_spec,
    normalize_pruning_spec,
    normalize_search_space,
    optimize_experiment,
    prepare_trial_config,
)
from src.experiments.support.notebook_lab import (
    build_analysis_frame_from_result,
    build_summary_frame,
    run_experiment_from_config,
)
from src.models.runtime import infer_feature_columns
from src.utils.config import load_experiment_config
from src.utils.config_validation import validate_resolved_config

pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 160)
SEED = 7
np.random.seed(SEED)
warnings.filterwarnings('ignore', category=FutureWarning)


In [ ]:
# Config: edit only these flags for normal experimentation.
OPTUNA_SPEC_PATH = repo_root / 'config/optuna/optuna_shock_meta_xgboost.yaml'
RUN_BASELINE = False
RUN_OPTUNA = False
RUN_BEST_TRIAL = False
BASELINE_SAMPLE_ROWS = 2_500
N_TRIALS_OVERRIDE = None  # example: 5 for a quick smoke run

with OPTUNA_SPEC_PATH.open('r', encoding='utf-8') as handle:
    OPTUNA_SPEC = yaml.safe_load(handle) or {}

BASE_CONFIG_PATH = repo_root / OPTUNA_SPEC['base_config']
BASE_CFG = load_experiment_config(BASE_CONFIG_PATH)
SEARCH_SPACE = normalize_search_space(OPTUNA_SPEC['search_space'])
OBJECTIVE = normalize_objective_spec(OPTUNA_SPEC.get('objective'))
PRUNING = normalize_pruning_spec(OPTUNA_SPEC.get('pruning'))
STUDY_CFG = dict(OPTUNA_SPEC.get('study', {}) or {})

validate_resolved_config(BASE_CFG)

print(f'Optuna spec: {OPTUNA_SPEC_PATH.relative_to(repo_root)}')
print(f'Base config:  {BASE_CONFIG_PATH.relative_to(repo_root)}')
print(f'Search dims:  {len(SEARCH_SPACE)}')
print(f'Objective:    {OBJECTIVE.metric_path} ({OBJECTIVE.direction})')
print(f'Pruning:      enabled={PRUNING.enabled}, metric={PRUNING.metric_path}')


In [ ]:
# Small display helpers
def show_df(title: str, frame: pd.DataFrame, rows: int = 20) -> pd.DataFrame:
    display(Markdown(f'### {title}'))
    if frame.empty:
        display(Markdown('_empty_'))
        return frame
    display(frame.head(rows))
    return frame


def flatten_dict(d: dict, prefix: str = '') -> dict:
    out = {}
    for key, value in dict(d or {}).items():
        name = f'{prefix}.{key}' if prefix else str(key)
        if isinstance(value, dict):
            out.update(flatten_dict(value, name))
        else:
            out[name] = value
    return out


def load_local_dukascopy_frame(cfg: dict) -> pd.DataFrame:
    data_cfg = dict(cfg.get('data', {}) or {})
    storage_cfg = dict(data_cfg.get('storage', {}) or {})
    load_path = storage_cfg.get('load_path')
    if not load_path:
        raise ValueError('Base config has no data.storage.load_path.')
    csv_path = repo_root / str(load_path)
    frame = pd.read_csv(csv_path)
    if 'timestamp' in frame.columns:
        ts_raw = frame['timestamp']
        if pd.api.types.is_numeric_dtype(ts_raw):
            unit = 'ms' if float(pd.to_numeric(ts_raw, errors='coerce').dropna().median()) > 1.0e11 else 's'
            timestamp = pd.to_datetime(ts_raw, unit=unit, utc=True).dt.tz_localize(None)
        else:
            timestamp = pd.to_datetime(ts_raw, utc=True, errors='coerce').dt.tz_localize(None)
        frame = frame.assign(timestamp=timestamp).set_index('timestamp')
    else:
        frame.index = pd.to_datetime(frame.index, utc=True, errors='coerce').tz_localize(None)
    frame = frame.sort_index()
    start = data_cfg.get('start')
    end = data_cfg.get('end')
    if start:
        frame = frame.loc[frame.index >= pd.to_datetime(start, utc=True).tz_localize(None)]
    if end:
        frame = frame.loc[frame.index <= pd.to_datetime(end, utc=True).tz_localize(None)]
    return frame.loc[:, ['open', 'high', 'low', 'close', 'volume']].copy()


def plot_series(frame: pd.DataFrame, columns: list[str], *, title: str, start=None, end=None, secondary_y: list[str] | None = None):
    cols = [col for col in columns if col in frame.columns]
    if not cols:
        print(f'No requested columns available for plot: {columns}')
        return None
    plot_frame = frame.loc[start:end, cols] if start is not None or end is not None else frame.loc[:, cols]
    ax = plot_frame.plot(figsize=(14, 4), grid=True, secondary_y=secondary_y or [])
    ax.set_title(title)
    plt.show()
    return ax


## Pipeline Map

This section shows the exact feature step indices used by `search_space.path`. Use it before editing any `features.<idx>.params...` path in the Optuna YAML.


In [ ]:
feature_steps = pd.DataFrame(
    [
        {
            'idx': idx,
            'step': step.get('step'),
            'enabled': step.get('enabled', True),
            'params': step.get('params', {}),
        }
        for idx, step in enumerate(BASE_CFG.get('features', []))
    ]
)
search_space_df = pd.DataFrame(
    [
        {
            'name': dim.name,
            'path': dim.path,
            'kind': dim.kind,
            'low': dim.low,
            'high': dim.high,
            'step': dim.step,
            'log': dim.log,
            'choices': list(dim.choices or []),
        }
        for dim in SEARCH_SPACE
    ]
)
objective_df = pd.DataFrame([flatten_dict(OPTUNA_SPEC.get('objective', {}))])
pruning_df = pd.DataFrame([flatten_dict(OPTUNA_SPEC.get('pruning', {}))])

show_df('Feature step index map', feature_steps, rows=len(feature_steps))
show_df('Optuna search space', search_space_df, rows=len(search_space_df))
show_df('Objective', objective_df)
show_df('Pruning', pruning_df)


## Feature Preview

This preview computes the same feature pipeline on a recent sample of the local BTCUSD CSV. It does not train the model. The purpose is to verify that selector-based model inputs still resolve after window-changing parameters.


In [ ]:
raw_frame = load_local_dukascopy_frame(BASE_CFG)
preview_raw = raw_frame.tail(BASELINE_SAMPLE_ROWS).copy()
preview_features = runner_mod._apply_feature_steps(preview_raw, list(BASE_CFG.get('features', []) or []))
model_cfg = dict(BASE_CFG.get('model', {}) or {})
resolved_feature_cols = infer_feature_columns(
    preview_features,
    explicit_cols=model_cfg.get('feature_cols'),
    feature_selectors=model_cfg.get('feature_selectors'),
)

feature_resolution_df = pd.DataFrame(
    [
        {
            'feature_col': col,
            'non_null': int(preview_features[col].notna().sum()),
            'coverage': float(preview_features[col].notna().mean()),
            'mean': float(preview_features[col].mean(skipna=True)) if pd.api.types.is_numeric_dtype(preview_features[col]) else np.nan,
            'std': float(preview_features[col].std(skipna=True)) if pd.api.types.is_numeric_dtype(preview_features[col]) else np.nan,
        }
        for col in resolved_feature_cols
    ]
)

print(f'Raw rows: {len(raw_frame):,} | Preview rows: {len(preview_raw):,} | Feature cols resolved: {len(resolved_feature_cols)}')
show_df('Resolved model feature coverage', feature_resolution_df.sort_values('coverage'), rows=len(feature_resolution_df))
preview_features.tail(5)


In [ ]:
# Visual checks: price, shock state, and selected model features.
plot_start = preview_features.index[-500] if len(preview_features) > 500 else None
plot_series(
    preview_features,
    ['close', 'shock_strength', 'shock_active_window'],
    title='Close vs shock strength / active window',
    start=plot_start,
    secondary_y=['shock_strength', 'shock_active_window'],
)
plot_series(
    preview_features,
    ['shock_ret_z_1h', 'shock_ret_z_4h', 'shock_atr_multiple_1h', 'shock_atr_multiple_4h'],
    title='Shock return z-scores and ATR multiples',
    start=plot_start,
)
plot_series(
    preview_features,
    [col for col in resolved_feature_cols if col.startswith(('regime_vol_ratio_', 'regime_absret_z_', 'bb_percent_b_', 'close_rsi_'))][:6],
    title='Selector-resolved regime / Bollinger / RSI features',
    start=plot_start,
)


## Trial Config Preview

Use this before running Optuna to verify that one sampled parameter set mutates only the intended config paths.


In [ ]:
manual_trial_params = {}
for dim in SEARCH_SPACE:
    if dim.kind == 'categorical':
        manual_trial_params[dim.name] = list(dim.choices or [])[0]
    elif dim.kind == 'int':
        manual_trial_params[dim.name] = int(dim.low)
    elif dim.kind == 'float':
        manual_trial_params[dim.name] = float(dim.low)
    else:
        manual_trial_params[dim.name] = False

trial_cfg = prepare_trial_config(
    BASE_CFG,
    trial_params=manual_trial_params,
    search_space=SEARCH_SPACE,
    logging_enabled=False,
)
validate_resolved_config(trial_cfg)

trial_preview_rows = []
for dim in SEARCH_SPACE:
    before = BASE_CFG
    after = trial_cfg
    for token in str(dim.path).split('.'):
        token = int(token) if token.isdigit() else token
        before = before[token]
        after = after[token]
    trial_preview_rows.append({'name': dim.name, 'path': dim.path, 'base_value': before, 'trial_value': after})

show_df('Manual first-choice trial mutation preview', pd.DataFrame(trial_preview_rows), rows=len(trial_preview_rows))


## Baseline Run

Set `RUN_BASELINE = True` in the config cell to run the base shock-meta XGBoost pipeline and build the analysis frame used by the plots below. This can take time because it runs walk-forward XGBoost and backtest logic.


In [ ]:
baseline_result = None
baseline_frame = pd.DataFrame()

if RUN_BASELINE:
    baseline_result = run_experiment_from_config(
        BASE_CFG,
        config_path=BASE_CONFIG_PATH,
        logging_enabled=False,
    )
    baseline_summary = build_summary_frame(baseline_result.evaluation.get('primary_summary', {}))
    baseline_frame = build_analysis_frame_from_result(baseline_result)
    show_df('Baseline primary summary', baseline_summary, rows=len(baseline_summary))
else:
    display(Markdown('Baseline run is disabled. Set `RUN_BASELINE = True` to execute it.'))


In [ ]:
if not baseline_frame.empty:
    oos = baseline_frame.loc[baseline_frame.get('oos_mask', True).astype(bool)].copy()
    plot_series(oos, ['strategy_equity'], title='Baseline OOS equity curve')
    plot_series(oos, ['strategy_drawdown'], title='Baseline OOS drawdown')
    plot_series(oos, ['strategy_positions', 'pred_prob'], title='Baseline position and predicted probability', secondary_y=['pred_prob'])
    plot_series(oos, ['strategy_net_returns', 'strategy_costs'], title='Baseline net returns and costs')
else:
    display(Markdown('No baseline plots yet. Run the baseline cell first.'))


## Optuna Study

Set `RUN_OPTUNA = True` only after the feature preview and trial config preview are clean. Start with `N_TRIALS_OVERRIDE = 3` or `5` for a smoke run, then increase slowly.


In [ ]:
study = None
trials_df = pd.DataFrame()

if RUN_OPTUNA:
    n_trials = int(N_TRIALS_OVERRIDE or STUDY_CFG.get('n_trials', 30))
    study = optimize_experiment(
        BASE_CONFIG_PATH,
        search_space=SEARCH_SPACE,
        objective=OBJECTIVE,
        pruning=PRUNING,
        study_name=STUDY_CFG.get('study_name'),
        storage=STUDY_CFG.get('storage'),
        load_if_exists=bool(STUDY_CFG.get('load_if_exists', False)),
        sampler=str(STUDY_CFG.get('sampler', 'tpe')),
        seed=STUDY_CFG.get('seed', 7),
        n_trials=n_trials,
        timeout=STUDY_CFG.get('timeout'),
        n_jobs=int(STUDY_CFG.get('n_jobs', 1)),
        logging_enabled=bool(STUDY_CFG.get('logging_enabled', False)),
        catch_exceptions=bool(STUDY_CFG.get('catch_exceptions', True)),
    )
    trials_df = study.trials_dataframe()
    show_df('Optuna trials', trials_df.sort_values('value', ascending=False), rows=min(len(trials_df), 25))
    show_df('Best trial params', pd.DataFrame([study.best_params]).T.rename(columns={0: 'value'}), rows=len(study.best_params))
    if study.best_trial.user_attrs.get('primary_summary'):
        show_df('Best trial primary summary', build_summary_frame(study.best_trial.user_attrs['primary_summary']))
else:
    display(Markdown('Optuna run is disabled. Set `RUN_OPTUNA = True` to execute it.'))


In [ ]:
if study is not None and not trials_df.empty:
    numeric_cols = [col for col in trials_df.columns if col.startswith('params_')]
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    trials_df[['number', 'value']].dropna().plot(x='number', y='value', marker='o', ax=axes[0], grid=True, legend=False)
    axes[0].set_title('Objective value by trial')
    if numeric_cols:
        corr = trials_df[numeric_cols + ['value']].apply(pd.to_numeric, errors='coerce').corr(numeric_only=True)['value'].drop('value').sort_values()
        corr.tail(15).plot(kind='barh', ax=axes[1], grid=True)
        axes[1].set_title('Top parameter correlations with objective')
    else:
        axes[1].axis('off')
    plt.tight_layout()
    plt.show()
else:
    display(Markdown('No Optuna study results yet.'))


## Best Trial Re-run

This cell materializes the best trial into the same base pipeline and optionally re-runs it. Keep this separate from the study run so you can inspect the chosen parameters before spending more time.


In [ ]:
best_trial_cfg = None
best_trial_result = None
best_trial_frame = pd.DataFrame()

if study is not None:
    best_trial_cfg = prepare_trial_config(
        BASE_CFG,
        trial_params=study.best_params,
        search_space=SEARCH_SPACE,
        logging_enabled=False,
    )
    validate_resolved_config(best_trial_cfg)
    show_df('Best trial feature steps', pd.DataFrame([
        {'idx': idx, 'step': step.get('step'), 'params': step.get('params', {})}
        for idx, step in enumerate(best_trial_cfg.get('features', []))
    ]), rows=len(best_trial_cfg.get('features', [])))

    if RUN_BEST_TRIAL:
        best_trial_result = run_experiment_from_config(
            best_trial_cfg,
            config_path=BASE_CONFIG_PATH,
            logging_enabled=False,
        )
        best_trial_frame = build_analysis_frame_from_result(best_trial_result)
        show_df('Best trial primary summary', build_summary_frame(best_trial_result.evaluation.get('primary_summary', {})))
else:
    display(Markdown('Run Optuna first to materialize a best-trial config.'))


In [ ]:
if not best_trial_frame.empty:
    oos = best_trial_frame.loc[best_trial_frame.get('oos_mask', True).astype(bool)].copy()
    plot_series(oos, ['strategy_equity'], title='Best trial OOS equity curve')
    plot_series(oos, ['strategy_drawdown'], title='Best trial OOS drawdown')
    plot_series(oos, ['strategy_positions', 'pred_prob'], title='Best trial position and predicted probability', secondary_y=['pred_prob'])
else:
    display(Markdown('No best-trial plots yet. Set `RUN_BEST_TRIAL = True` after an Optuna run.'))


## Notes

- If many trials fail with selector errors, reduce the search space before expanding it.
- If best trials have low `derived.trade_count`, increase the trade-count penalty or narrow signal thresholds.
- If fold Sharpe dispersion is high, increase `objective.stability_weight` or shrink the model parameter space.
- The next production step is to promote only a small, stable set of tuned params back into a separate experiment config and re-test OOS.
